In [3]:
import pandas as pd
import mlflow
import mlflow.sklearn

In [4]:
df_X = pd.read_csv("df_X_clean.csv", index_col=0)
df_Y = pd.read_csv("Y_train_CVw08PX.csv", index_col=0)

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

X_train, X_test, y_train, y_test = train_test_split(
    df_X["text"],
    df_Y["prdtypecode"],
    test_size=0.2,
    random_state=42
)

In [6]:
mlflow.set_experiment("rakuten-baseline")

with mlflow.start_run():
    # Paramètres
    max_iter = 1000
    test_size = 0.2
    
    # Entraînement
    tfidf = TfidfVectorizer()
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)
    
    model = LogisticRegression(max_iter=max_iter)
    model.fit(X_train_tfidf, y_train)
    
    # Évaluation
    y_pred = model.predict(X_test_tfidf)
    score = f1_score(y_test, y_pred, average="weighted")
    
    # Log MLflow
    mlflow.log_param("max_iter", max_iter)
    mlflow.log_param("test_size", test_size)
    mlflow.log_metric("weighted_f1", score)
    mlflow.sklearn.log_model(model, "model")
    
    print(f"Weighted F1 : {score}")

2026/05/15 17:21:20 INFO mlflow.tracking.fluent: Experiment with name 'rakuten-baseline' does not exist. Creating a new experiment.


Weighted F1 : 0.8040315407547788


In [8]:
configs = [
    {"max_iter": 1000, "max_features": 50000},
    {"max_iter": 1000, "max_features": 100000},
    {"max_iter": 1000, "max_features": None},
]

In [9]:
for config in configs:
    with mlflow.start_run():
        tfidf = TfidfVectorizer(max_features=config["max_features"])
        X_train_tfidf = tfidf.fit_transform(X_train)
        X_test_tfidf = tfidf.transform(X_test)
        
        model = LogisticRegression(max_iter=config["max_iter"])
        model.fit(X_train_tfidf, y_train)
        
        y_pred = model.predict(X_test_tfidf)
        score = f1_score(y_test, y_pred, average="weighted")
        
        mlflow.log_param("max_features", config["max_features"])
        mlflow.log_param("max_iter", config["max_iter"])
        mlflow.log_metric("weighted_f1", score)
        
        print(f"max_features={config['max_features']} → F1={score:.4f}")

max_features=50000 → F1=0.8056
max_features=100000 → F1=0.8051
max_features=None → F1=0.8040


In [10]:
configs2 = [
    {"max_features": 50000, "ngram_range": (1,1)},
    {"max_features": 50000, "ngram_range": (1,2)},
    {"max_features": 50000, "ngram_range": (1,3)},
]

for config in configs2:
    with mlflow.start_run():
        tfidf = TfidfVectorizer(
            max_features=config["max_features"],
            ngram_range=config["ngram_range"]
        )
        X_train_tfidf = tfidf.fit_transform(X_train)
        X_test_tfidf = tfidf.transform(X_test)
        
        model = LogisticRegression(max_iter=1000)
        model.fit(X_train_tfidf, y_train)
        
        y_pred = model.predict(X_test_tfidf)
        score = f1_score(y_test, y_pred, average="weighted")
        
        mlflow.log_param("max_features", config["max_features"])
        mlflow.log_param("ngram_range", config["ngram_range"])
        mlflow.log_metric("weighted_f1", score)
        
        print(f"ngram_range={config['ngram_range']} → F1={score:.4f}")

ngram_range=(1, 1) → F1=0.8056
ngram_range=(1, 2) → F1=0.8046
ngram_range=(1, 3) → F1=0.8028
